# Monitoring with Grafana Tempo

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pecca-core/pecca/blob/main/examples/notebooks/monitoring_grafana_tempo.ipynb)

Spans go to **Tempo**; **Grafana** shows them in a dashboard that is provisioned from `examples/monitoring/grafana/`. We query Tempo with TraceQL and check the dashboard through Grafana's API.
> Needs **Docker** for the container cells; without it (`PECCA_SKIP_DOCKER=1`) those cells are skipped and a fallback runs.

In [1]:
%pip install -q pecca requests

In [2]:
import json, os, re, shutil, subprocess, tempfile, time
from pathlib import Path

import pandas as pd
import pecca
from pecca.core import context
from pecca.data.dataset import Dataset, canonicalize
from pecca.data.synthetic import generate

HERE = Path.cwd()                                   # the notebook's folder (examples/notebooks in the repo)
work = tempfile.mkdtemp()
os.chdir(work)
os.environ["PECCA_HOME"] = os.path.join(work, ".pecca")
os.environ["TQDM_DISABLE"] = "1"
os.environ["MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT"] = "1"
import logging
try:
    import mlflow  # MLflow resets its logger level on import, so quiet it afterwards
    logging.getLogger("mlflow").setLevel(logging.ERROR)
except ImportError:
    pass

# Docker-backed cells run only when Docker is reachable and PECCA_SKIP_DOCKER != 1 (CI sets it).
USE_DOCKER = (os.environ.get("PECCA_SKIP_DOCKER") != "1" and shutil.which("docker") is not None
              and subprocess.run(["docker", "info"], capture_output=True).returncode == 0)
print("docker:", "available" if USE_DOCKER else "skipped (PECCA_SKIP_DOCKER=1 or Docker not reachable)")

def sh(*args, check=True):
    return subprocess.run([str(a) for a in args], capture_output=True, text=True, check=check)

import atexit
def cleanup_later(*cmd):
    """Run `cmd` when the kernel exits, so containers are removed even if a cell fails."""
    atexit.register(lambda: subprocess.run([str(c) for c in cmd], capture_output=True))

def free_port():
    import socket
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

def wait_for(check, what, seconds=90):
    end = time.time() + seconds
    while time.time() < end:
        try:
            if check():
                return
        except Exception:
            pass
        time.sleep(1)
    raise TimeoutError(f"{what} did not become ready in {seconds}s")

def train_demo(call="route_rfi", rows=3000, candidates=("tfidf_linear",), **kw):
    """Train a small model on the synthetic demo data (LLM answers + human corrections)."""
    df = generate(rows)
    df["text"] = df.email_subject + "\n\n" + df.email_body
    cols = {"input": "text", "llm_output": "llm_rfi", "human_label": "human_rfi", "call_name": {"literal": call.split("/")[-1]}}
    return pecca.train(call, Dataset(canonicalize(df, cols), cols), candidates=list(candidates), **kw)

docker: available


## 1. Start Tempo + Grafana
Grafana: http://localhost:3000 (anonymous admin, local demo only).

In [3]:
COMPOSE = ["docker", "compose", "-p", "pecca-tempo", "-f", str(HERE.parent / "monitoring" / "tempo" / "docker-compose.yml")]
if USE_DOCKER:
    import requests
    cleanup_later(*COMPOSE, "down", "-v")
    sh(*COMPOSE, "up", "-d")
    wait_for(lambda: requests.get("http://127.0.0.1:3200/ready", timeout=2).text.strip() == "ready", "Tempo", 120)
    wait_for(lambda: requests.get("http://127.0.0.1:3000/api/health", timeout=2).json().get("database") == "ok", "Grafana", 120)
    print("Tempo and Grafana are up")

Tempo and Grafana are up


## 2. Run shadow calls with telemetry on

In [4]:
telemetry = "  telemetry: {exporter: otlp, endpoint: http://127.0.0.1:4318/v1/traces, service: pecca-demo}\n" if USE_DOCKER else ""
Path("pecca.yaml").write_text("version: 1\nworkspace:\n  name: default\n" + telemetry + "projects:\n  default: {governance: {extends: none}}\n")
context.reset()
print(Path("pecca.yaml").read_text())

version: 1
workspace:
  name: default
  telemetry: {exporter: otlp, endpoint: http://127.0.0.1:4318/v1/traces, service: pecca-demo}
projects:
  default: {governance: {extends: none}}



In [5]:
result = train_demo()
print(f"trained {result.winner}: macro_f1 {result.metric:.2f}")

trained tfidf_linear: macro_f1 0.92


In [6]:
# The "LLM": replay the simulated LLM's answers for a few new emails (no API needed).
new = generate(30, seed=11)
texts = (new.email_subject + "\n\n" + new.email_body).tolist()
said = dict(zip(texts, new.llm_rfi))

@pecca.replace("route_rfi", mode="shadow")
def route_rfi(text: str) -> str:
    return said[text]                      # stand-in for my_llm.classify(text), unchanged

pecca.promote("route_rfi", "shadow")
for t in texts:
    route_rfi(t)

from opentelemetry import trace
trace.get_tracer_provider().force_flush() if hasattr(trace.get_tracer_provider(), "force_flush") else None
from pecca.runtime.logging_sink import flush_all
flush_all()

print(f"ran {len(texts)} shadow calls")

ran 30 shadow calls


## 3. Query Tempo with TraceQL
The same queries back the provisioned dashboard panels.

In [7]:
def traceql(q):
    r = requests.get("http://127.0.0.1:3200/api/search", params={"q": q, "limit": 100}, timeout=10).json()
    return r.get("traces", [])

if USE_DOCKER:
    wait_for(lambda: len(traceql('{ name = "pecca.call" }')) >= len(texts), "spans in Tempo", 90)
    queries = {
        "all pecca.call spans": '{ name = "pecca.call" }',
        "served by the LLM (shadow)": '{ span.pecca.served_by = "llm" }',
        "model disagreed with the LLM": '{ span.pecca.agreement = false }',
        "served by fallback": '{ span.pecca.served_by = "fallback" }',
        "confidence < 0.6": '{ name = "pecca.call" && span.pecca.confidence < 0.6 }',
    }
    print(pd.Series({k: len(traceql(q)) for k, q in queries.items()}, name="traces").to_frame())
else:
    print("skipped: needs Docker.")

                              traces
all pecca.call spans              30
served by the LLM (shadow)        30
model disagreed with the LLM       6
served by fallback                 0
confidence < 0.6                   0


## 4. The provisioned Grafana dashboard

In [8]:
if USE_DOCKER:
    dash = requests.get("http://127.0.0.1:3000/api/dashboards/uid/pecca-llm-replacement", timeout=10).json()["dashboard"]
    via_grafana = requests.get("http://127.0.0.1:3000/api/datasources/proxy/uid/pecca-tempo/api/search",
                               params={"q": '{ name = "pecca.call" }', "limit": 100}, timeout=10).json()["traces"]
    print("dashboard:", dash["title"], f"| Grafana -> Tempo datasource returns {len(via_grafana)} traces")
    for p in dash["panels"]:
        print(f" - {p['title']:<28} {p['targets'][0]['query']}")
    print("Open http://localhost:3000/d/pecca-llm-replacement to see it.")
else:
    d = json.loads((HERE.parent / "monitoring" / "grafana" / "dashboards" / "pecca-llm-replacement.json").read_text())
    print("dashboard JSON (not started):", d["title"], [p["title"] for p in d["panels"]])

dashboard: Pecca: LLM replacement | Grafana -> Tempo datasource returns 30 traces
 - Recent Pecca calls           { name = "pecca.call" }
 - Served by the LLM fallback   { name = "pecca.call" && span.pecca.served_by = "fallback" }
 - Shadow disagreements         { name = "pecca.call" && span.pecca.agreement = false }
 - Low confidence (< 0.6)       { name = "pecca.call" && span.pecca.confidence < 0.6 }
Open http://localhost:3000/d/pecca-llm-replacement to see it.


## 5. Clean up

In [9]:
if USE_DOCKER:
    sh(*COMPOSE, "down", "-v")
    print("Tempo and Grafana stopped")

Tempo and Grafana stopped
